# BreadboardAOI — Qwen3-VL tutor runtime (Colab)

This notebook turns a Colab GPU runtime into the **cognitive loop** of the BreadboardAOI workstation. Vision, motion gating and the KiCad DRC engine stay on your PC at 60 FPS; only the Qwen3-VL diagnosis is offloaded here.

## One-time setup

1. In BreadboardAOI open the **REMOTE AI** tab, paste your Google OAuth **Desktop app** client ID and press **LINK GOOGLE**.
2. Press **PUBLISH SESSION**. The app writes a one-time bearer key to `MyDrive/breadboardaoi-colab.json`.
3. Here: **Runtime → Change runtime type → GPU**, then **Run all**.
4. This notebook publishes its tunnel URL back into the same Drive file. The app switches over within 15 seconds, or press **REFRESH ENDPOINT**.

## Models are GitHub-only

To respect the project's download policy, weights are pulled from this repository's `models-v1` GitHub Release instead of Hugging Face or another CDN. That release is prepared by `.github/workflows/publish-models.yml`, which mirrors the official Qwen3-VL-8B-Instruct GGUF and splits it below GitHub's 2 GB per-file limit (`part_00`, `part_01`, ...).

> Colab's terms restrict always-on, non-interactive serving. Treat this as an interactive lab session: idle runtimes are reclaimed, and heavy use may be throttled.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------------
MODEL_FILE = "Qwen3VL-8B-Instruct-Q4_K_M.gguf"
MMPROJ_FILE = "mmproj-Qwen3VL-8B-Instruct-F16.gguf"

# GitHub Release assets only (no HuggingFace, no vendor CDNs at runtime).
# The models-v1 release is created by .github/workflows/publish-models.yml,
# which mirrors the official weights and splits the GGUF below 2 GB per asset.
RELEASE_BASE = "https://github.com/wallali77666-cyber/BreadBoard-Assistant/releases/download/models-v1"
GITHUB_MODEL_URL = ""
MODEL_PARTS = [
    f"{RELEASE_BASE}/{MODEL_FILE}.part_00",
    f"{RELEASE_BASE}/{MODEL_FILE}.part_01",
    f"{RELEASE_BASE}/{MODEL_FILE}.part_02",
]
GITHUB_MMPROJ_URL = f"{RELEASE_BASE}/{MMPROJ_FILE}"

SESSION_FILE = "/content/drive/MyDrive/breadboardaoi-colab.json"
PORT = 8089

# 8B Q4_K_M fits a free T4 (16 GB VRAM) and runs fully on the GPU.
# The 30B-A3B needs Colab Pro (L4/A100) or the local RTX 3060 workstation.


In [ ]:
# ---------------------------------------------------------------------------
# 2. Link this runtime to your Google account and read the shared session key
# ---------------------------------------------------------------------------
from google.colab import auth, drive
import json, os

auth.authenticate_user()
drive.mount("/content/drive")

if not os.path.exists(SESSION_FILE):
    raise SystemExit(
        "No session file found. In BreadboardAOI open the REMOTE AI tab, press "
        "PUBLISH SESSION, wait a few seconds, then re-run this cell."
    )

with open(SESSION_FILE) as handle:
    session = json.load(handle)

API_KEY = session.get("apiKey")
if not API_KEY:
    raise SystemExit("The session file has no apiKey. Press PUBLISH SESSION again in the app.")

print("session linked:", API_KEY[:10] + "...")
print("current status:", session.get("status"))

In [ ]:
# ---------------------------------------------------------------------------
# 3. Build llama-server with CUDA from GitHub source (~5-8 min on a T4)
# ---------------------------------------------------------------------------
!apt-get -qq update
!apt-get -qq install -y libcurl4-openssl-dev > /dev/null
!rm -rf /content/llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF
!cmake --build /content/llama.cpp/build --config Release -j 8 --target llama-server

In [ ]:
# ---------------------------------------------------------------------------
# 4. Fetch the weights from GitHub Releases (with multi-part concatenation)
# ---------------------------------------------------------------------------
import os, shutil, urllib.request

os.makedirs("/content/models", exist_ok=True)
model_path = f"/content/models/{MODEL_FILE}"
mmproj_path = f"/content/models/{MMPROJ_FILE}"

def download(url: str, dest: str) -> None:
    print("GET", url)
    urllib.request.urlretrieve(url, dest)

if not os.path.exists(model_path):
    if GITHUB_MODEL_URL:
        download(GITHUB_MODEL_URL, model_path)
    elif MODEL_PARTS:
        parts = []
        for index, url in enumerate(MODEL_PARTS):
            part = f"{model_path}.part{index:02d}"
            download(url, part)
            parts.append(part)
        with open(model_path, "wb") as output:
            for part in parts:
                with open(part, "rb") as source:
                    shutil.copyfileobj(source, output)
                os.remove(part)
    else:
        raise SystemExit(
            "No model URL configured. Upload the GGUF to a GitHub Release and set "
            "GITHUB_MODEL_URL or MODEL_PARTS at the top of this notebook."
        )

if not os.path.exists(mmproj_path):
    if GITHUB_MMPROJ_URL:
        download(GITHUB_MMPROJ_URL, mmproj_path)
    else:
        raise SystemExit(
            "No mmproj configured. Vision prompts need the projector: set GITHUB_MMPROJ_URL."
        )

print("model:", model_path, os.path.getsize(model_path) // (1024 * 1024), "MB")
print("mmproj:", mmproj_path, os.path.getsize(mmproj_path) // (1024 * 1024), "MB")

In [ ]:
# ---------------------------------------------------------------------------
# 5. Serve llama-server through a cloudflared tunnel and publish the endpoint
# ---------------------------------------------------------------------------
import json, os, re, subprocess, time, urllib.request
from datetime import datetime, timezone

SERVER = "/content/llama.cpp/build/bin/llama-server"
server_args = [SERVER, "--model", model_path]
if os.path.exists(mmproj_path):
    server_args += ["--mmproj", mmproj_path]
server_args += [
    "--host", "127.0.0.1", "--port", str(PORT),
    "--n-gpu-layers", "99",
    "--ctx-size", "4096",
    "--threads", "4", "--threads-batch", "8",
    "--flash-attn",
    "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
    "--api-key", API_KEY,
]

server_log = open("/content/llama-server.log", "w")
server = subprocess.Popen(server_args, stdout=server_log, stderr=subprocess.STDOUT)
print("llama-server pid", server.pid)

def healthy() -> bool:
    # llama-server answers 503 while weights load and 200 once ready. With
    # --api-key the bearer token is checked on every route, so send it here too.
    probe = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/health",
        headers={"Authorization": f"Bearer {API_KEY}"},
    )
    try:
        with urllib.request.urlopen(probe, timeout=3) as response:
            return response.status == 200
    except Exception:
        return False

for _ in range(180):
    if healthy():
        print("llama-server is ready")
        break
    if server.poll() is not None:
        raise SystemExit("llama-server exited early; read /content/llama-server.log")
    time.sleep(2)
else:
    raise SystemExit("llama-server did not become ready within 6 minutes")

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True,
        check=True,
    )

open("/content/cloudflared.log", "w").close()
tunnel = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared", "tunnel",
        "--url", f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
        "--logfile", "/content/cloudflared.log",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

tunnel_url = None
for _ in range(60):
    with open("/content/cloudflared.log") as handle:
        text = handle.read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(2)

if not tunnel_url:
    raise SystemExit("cloudflared did not publish a URL; read /content/cloudflared.log")

print("tunnel:", tunnel_url)

request = urllib.request.Request(tunnel_url + "/health", headers={"Authorization": f"Bearer {API_KEY}"})
for _ in range(20):
    try:
        urllib.request.urlopen(request, timeout=10)
        break
    except Exception as exc:
        print("waiting for the public tunnel:", exc)
        time.sleep(3)

try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).strip()
except Exception:
    gpu = "Colab runtime"

session.update(
    {
        "status": "serving",
        "endpointUrl": tunnel_url,
        "apiKey": API_KEY,
        "model": MODEL_FILE,
        "gpu": gpu,
        "detail": f"llama-server :{PORT} via cloudflared quick tunnel",
        "updatedUtc": datetime.now(timezone.utc).isoformat(),
    }
)

with open(SESSION_FILE, "w") as handle:
    json.dump(session, handle, indent=2)

print("published:", SESSION_FILE)
print("BreadboardAOI will switch to this runtime within 15 seconds.")

In [ ]:
# ---------------------------------------------------------------------------
# 6. Keep the session alive. Stop this cell (or the runtime) to end it.
# ---------------------------------------------------------------------------
import json, time
from datetime import datetime, timezone

try:
    while True:
        time.sleep(60)
        server.poll()
        if server.returncode is not None:
            session.update({"status": "stopped", "detail": "llama-server exited"})
            with open(SESSION_FILE, "w") as handle:
                json.dump(session, handle, indent=2)
            print("llama-server exited; session marked stopped")
            break
        print("alive", datetime.now(timezone.utc).strftime("%H:%M:%S UTC"))
except KeyboardInterrupt:
    session.update(
        {
            "status": "stopped",
            "detail": "notebook stopped",
            "updatedUtc": datetime.now(timezone.utc).isoformat(),
        }
    )
    with open(SESSION_FILE, "w") as handle:
        json.dump(session, handle, indent=2)
    print("session marked stopped")

## Troubleshooting

| Symptom | Fix |
| :--- | :--- |
| App shows `Colab session is 'waiting'` | Cell 5 has not reached the publish step yet, or the notebook is signed into a different Google account. |
| App shows `KEY REJECTED (401)` | The Drive file was overwritten with a new key. Press **REFRESH ENDPOINT** in the app. |
| `llama-server exited early` | Read `/content/llama-server.log`; usually a missing mmproj or an out-of-memory GPU. |
| Tunnel URL works in the browser but not in the app | Keep the `--api-key` flag; the app sends the same bearer key from the Drive file. |
| Nothing in `MyDrive` | The notebook account must match the account linked in the app. |

### Runtime sizes

- **Free T4 (16 GB VRAM / 12.7 GB RAM):** Qwen3-VL-8B Q4_K_M + mmproj is comfortable. A 30B-A3B MoE does not fit.
- **Colab Pro (A100 40 GB):** Qwen3-VL-30B-A3B Q4_K_M fits with `--n-gpu-layers 99 --n-cpu-moe 22`, mirroring the local command line.

The desktop app keeps a **local llama-server** and the rule-based tutor as automatic fallbacks, so closing this tab never breaks inspection.